# alchemy-viz in a notebook

`view(obj)` draws a gufe or OpenFE object in the cell: a ligand, a protein, a
ligand network, a whole campaign. There is nothing to convert first and no
alchemy-viz object to build - `view()` takes what the planner handed you.

```
pip install alchemy-viz[notebook]
```

Run the cell below. Everything from here to **Reference** is usage; the
environment check, the sizes and the failure panels live at the bottom, where
they stop being in the way once you have read them once.

In [ ]:
import json
from pathlib import Path

from alchemy_viz import view

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples").is_dir())
EXAMPLES = root / "examples"
payloads = {p.stem: json.loads(p.read_text()) for p in sorted(EXAMPLES.glob("*.json"))}

# Ten TYK2 ligands and the nine mappings OpenFE's RBFE tutorial plans between
# them. Click an edge: its atom mapping is drawn on the right.
view(payloads["ligand_network_medium"])

---

# Your own objects

Three places an OpenFE object lives, and the one call that draws each:

```python
from gufe import LigandNetwork
from gufe.tokenization import GufeTokenizable

view(LigandNetwork.from_graphml(Path("network_setup/ligand_network.graphml").read_text()))
view(GufeTokenizable.from_json("network_setup/network_setup.json"))
view(ligand)  # or any gufe object already in the session
```

There is no alchemy-viz-specific API and nothing openfe-specific to learn:
`openfe.SmallMoleculeComponent` **is** `gufe.SmallMoleculeComponent`, the same
class object, and `view()` dispatches on the gufe classes. The three sections
below are the three lines above in full.

## 1. A directory written by `openfe plan-rbfe-network`

The common case: you planned a campaign with the OpenFE command line and want to
look at what it produced before spending any CPU on it. The planner writes three
things, and `view()` reads all three.

```
$ openfe plan-rbfe-network -M ligands.sdf -p protein.pdb -o network_setup

network_setup/
  network_setup.json           the AlchemicalNetwork - the whole campaign
  ligand_network.graphml       the LigandNetwork - the ligands and the mappings
  transformations/
    easy_rbfe_lig_ejm_31_solvent_lig_ejm_42_solvent.json     one edge, one file
    ...
```

Note the campaign's filename: it is named after the output directory, not
`alchemical_network.json`. `openfe plan-rhfe-network` writes the same layout.

In [ ]:
# Point this at your own planner output. If it is not there, the cells below fall
# back to a network that came out of exactly this command: scripts/data/tyk2_network.graphml
# is the ligand_network.graphml from OpenFE's RBFE tutorial, committed here
# because neither the ligands nor the planner is a dependency of this repository.
OPENFE_SETUP = Path("network_setup")

if OPENFE_SETUP.is_dir():
    print(f"reading {OPENFE_SETUP}/")
    for item in sorted(OPENFE_SETUP.rglob("*.json")) + sorted(OPENFE_SETUP.rglob("*.graphml")):
        print(f"  {item.relative_to(OPENFE_SETUP)}  ({item.stat().st_size:,} bytes)")
else:
    print(f"no {OPENFE_SETUP}/ here - falling back to the tutorial's own network.")

In [ ]:
# The ligand network. `LigandNetwork.from_graphml` is gufe's reader and openfe's
# writer, so this is the file the planner wrote, not a copy of it.
from gufe import LigandNetwork

graphml = OPENFE_SETUP / "ligand_network.graphml"
if not graphml.is_file():
    graphml = root / "scripts" / "data" / "tyk2_network.graphml"

network = LigandNetwork.from_graphml(graphml.read_text(encoding="utf-8"))
print(f"{graphml.name}: {len(network.nodes)} ligands, {len(network.edges)} mappings")

view(network)

In [ ]:
# The campaign, and one edge of it. Both are written by gufe's `to_json`, so both
# are read by `from_json` - which tries the keyed chain first and falls back to
# the dict form, so it does not matter which of gufe's shapes a given file is in.
from gufe.tokenization import GufeTokenizable

campaign = OPENFE_SETUP / f"{OPENFE_SETUP.name}.json"
edges = sorted((OPENFE_SETUP / "transformations").glob("*.json")) if OPENFE_SETUP.is_dir() else []

if campaign.is_file():
    alchemical_network = GufeTokenizable.from_json(campaign)
    print(f"{campaign.name}: {len(alchemical_network.edges)} transformations")
    display(view(alchemical_network))
if edges:
    transformation = GufeTokenizable.from_json(edges[0])
    print(f"{edges[0].name}: {transformation.name}")
    display(view(transformation))
if not campaign.is_file():
    print("Set OPENFE_SETUP above to a real planner output directory to run this cell.")
    print("The committed examples/alchemical_network_medium.json is the same TYK2 campaign,")
    print("already serialized, if you only want to see what the view looks like:")
    display(view(payloads["alchemical_network_medium"]))

## 2. Planning one in Python instead

The same campaign, built in a session rather than on the command line. This is
OpenFE's API rather than ours, and the point of showing it is where `view()` goes:
**after the planner and before the protocol**, which is the moment a bad mapping
is still cheap to fix.

In [ ]:
# Guarded, because openfe and its mappers are not installed here. Copy it into an
# openfe environment and it runs as written.
try:
    import openfe
    from openfe.setup.ligand_network_planning import generate_minimal_spanning_network
    from rdkit import Chem

    ligands = [openfe.SmallMoleculeComponent(mol) for mol in Chem.SDMolSupplier("ligands.sdf", removeHs=False)]
    protein = openfe.ProteinComponent.from_pdb_file("protein.pdb")
    solvent = openfe.SolventComponent()

    planned = generate_minimal_spanning_network(
        ligands=ligands,
        mappers=[openfe.LomapAtomMapper()],
        scorer=openfe.lomap_scorers.default_lomap_score,
    )
    display(view(planned))  # look at the mappings before committing to them

    campaign = openfe.setup.RBFEAlchemicalNetworkPlanner()(
        ligands=ligands,
        solvent=solvent,
        protein=protein,
    )
    display(view(campaign))  # and at the campaign the planner builds from them
except ImportError as e:
    print(f"skipped: {e}")
    print()
    print("The call that matters is the shape of it: view() takes whatever the planner")
    print("returned, with no conversion step in between.")

## 3. From the shell, with no Python at all

`alchemy-viz` reads the planner's files directly, so a campaign can be turned into
a page you can mail to someone without opening a notebook:

```bash
alchemy-viz network_setup/network_setup.json -o campaign.html
alchemy-viz network_setup/transformations/easy_rbfe_lig_ejm_31_solvent_lig_ejm_42_solvent.json
```

One self-contained HTML file per object. It is not `openfe view`, and nothing in
openfe calls it: alchemy-viz is a separate package that reads openfe's output
using the gufe that is already in the environment.

---

# Every view, on the committed examples

One cell per fixture in `examples/` - the same files pytest, vitest and the
drag-and-drop dev page read - so you can re-run just the one you are working on.
Every type the schema declares has a view; what differs between them is how much
there is in the payload to draw.

[`alchemy-viz-gallery.ipynb`](./alchemy-viz-gallery.ipynb) is this same set as
screenshots, which is the one to look at on GitHub, where iframes are stripped
from cell outputs.

### `small_molecule.json`

2D RDKit depiction beside the 3D conformer, with SMILES, charge and atom counts.

In [ ]:
view(payloads["small_molecule"])

### `small_molecule_charged.json`

The same view with a non-zero formal charge.

In [ ]:
view(payloads["small_molecule_charged"])

### `protein.json`

3Dmol with representation and colour-scheme switchers; waters hidden by default.

In [ ]:
view(payloads["protein"])

### `protein_fragment.json`

A small protein, so the 3D view loads fast while iterating.

In [ ]:
view(payloads["protein_fragment"])

### `protein_membrane.json`

The same element, reached the other way. Python's MRO walk gives a membrane
system the protein *builder*, but the payload it emits says
`ProteinMembraneComponentViz`, and that type needs its own entry in the browser's
dispatch table - inheritance on one side of the contract is not inheritance on
the other.

In [ ]:
view(payloads["protein_membrane"])

### `solvated_pdb.json`

`SolvatedPDBComponentViz`, dispatched to the same element for the same reason.

In [ ]:
view(payloads["solvated_pdb"])

### `ligand_network.json`

Radial graph of the ligands, atom mapping on the right for the selected edge.

In [ ]:
view(payloads["ligand_network"])

### `ligand_network_named.json`

The same network with the ligands named, so labels replace gufe keys.

In [ ]:
view(payloads["ligand_network_named"])

### `chemical_system.json`

The system's components down the left, the selected one drawn on the right in
whichever view its own type gets - so a chemical system is a chooser over the
views above rather than a picture of its own.

In [ ]:
view(payloads["chemical_system"])

### `ligand_atom_mapping.json`

One mapping on its own, in the same element the ligand network's detail pane
mounts. All three cards open on plain 3D; the correspondence itself is drawn by
the other modes in the switcher, `3D-Map`, `Pairs` and `2D`.

In [ ]:
view(payloads["ligand_atom_mapping"])

### `solvent.json`

A solvent component is a specification rather than a structure, so its view is a
settings card beside a schematic of which ions are present - and it says under
itself, in words, that it is not showing how many.

In [ ]:
view(payloads["solvent"])

---

# From live gufe objects

The same call, handed real objects rather than payload JSON. `view()` serializes
through `alchemy_viz.payload_for`, so this is the path a user actually takes.

In [ ]:
import warnings

import gufe

warnings.filterwarnings("ignore", message=".*hydrogen atoms.*")

data = Path(gufe.__file__).parent / "tests" / "data"

net = gufe.LigandNetwork.from_graphml((data / "ligand_network.graphml").read_text())
edge = sorted(net.edges, key=lambda e: str(e.key))[0]
ligand_A, ligand_B = edge.componentA, edge.componentB

protein = gufe.ProteinComponent.from_pdb_file(str(data / "181l.pdb"), name="T4 lysozyme")
solvent = gufe.SolventComponent()

stateA = gufe.ChemicalSystem({"ligand": ligand_A, "protein": protein, "solvent": solvent}, name="complex A")
stateB = gufe.ChemicalSystem({"ligand": ligand_B, "protein": protein, "solvent": solvent}, name="complex B")

# A Transformation needs a Protocol, and gufe's own dummy is the one that needs
# no simulation engine installed. It lives under gufe's tests, so treat it as
# optional rather than as something to fail the notebook over.
try:
    from gufe.tests.test_protocol import DummyProtocol

    protocol = DummyProtocol(settings=DummyProtocol.default_settings())
    transformation = gufe.Transformation(stateA=stateA, stateB=stateB, protocol=protocol, mapping=edge, name="A to B")
    alchemical_net = gufe.AlchemicalNetwork(edges=[transformation], name="demo campaign")
except ImportError as e:
    transformation = alchemical_net = None
    print(f"no DummyProtocol ({e}) - the transformation cells will be skipped")

label = lambda component: component.name or str(component.key)  # noqa: E731
print(f"ligands   {label(ligand_A)} -> {label(ligand_B)}")
print(f"network   {len(net.nodes)} ligands, {len(net.edges)} edges")

### `SmallMoleculeComponent`

In [ ]:
view(ligand_A)

### `ProteinComponent`, with a size override

In [ ]:
view(protein, height="500px")

### `LigandNetwork`

In [ ]:
view(net)

### `ChemicalSystem`, `LigandAtomMapping`, `SolventComponent`

All three reach the panel today.

In [ ]:
view(stateA)

### `Transformation` and `AlchemicalNetwork`

In [ ]:
view(transformation) if transformation is not None else "no DummyProtocol - skipped"

In [ ]:
view(alchemical_net) if alchemical_net is not None else "no DummyProtocol - skipped"

---

# Every way of delivering one

## 1. Static only

`live=False` skips anywidget even when it is installed. This is exactly what a
reader with no kernel sees, so it is the honest preview of an export.

In [ ]:
view(ligand_A, live=False, height="420px")

## 2. Live, and updated in place

Run the next cell, then the one after it, and **watch the output above change**
without re-running it. That is the whole point of the live layer: the element's
update beat, driven from Python.

In [ ]:
w = view(ligand_A, height="420px")
w

In [ ]:
w.payload = ligand_B  # scroll up: the cell above is now showing ligand B
print("showing:", w.payload.get("name") or w.payload["type"])

## 3. Live, without the exported copy

`static=False` drops the `text/html` layer. The cell halves what it costs to
display, and an exported notebook has a hole where it was. *What a view costs*,
under **Reference**, is why the knob exists.

In [ ]:
view(net, static=False, height="420px")

## 4. The file the CLI writes

`to_html` returns a string and writes nothing; the CLI is one answer to where it
goes, and so is this cell.

In [ ]:
import tempfile

from alchemy_viz import to_html

scratch = Path(tempfile.gettempdir())

out = scratch / "protein.html"
out.write_text(to_html(payloads["protein"]), encoding="utf-8")
print(f"wrote {out} ({out.stat().st_size:,} bytes) - open it in a browser")

## 5. ...via the CLI itself

The same page, from the shell. `alchemy-viz` is a development convenience rather
than the OpenFE CLI integration, but it is the same `to_html` underneath.

In [ ]:
source = EXAMPLES / "small_molecule.json"
target = scratch / "small_molecule.html"
!alchemy-viz "{source}" -o "{target}"

---

# Reference

Worth reading once: what comes out of a `view()` call, whether this environment
has what the live layer needs, what a view costs to display, and what a payload
that cannot be drawn looks like.

## What a cell gets

Two layers come out of one `view()` call:

| layer | mimetype | needs | gives |
| --- | --- | --- | --- |
| static | `text/html` - the page in an `<iframe srcdoc>` | nothing | a saved notebook that still draws with no kernel |
| live | a widget view - shell page + payload as widget state | `anywidget` | `w.payload = other` redraws in place |

Your frontend picks. With a live kernel you get the widget; `nbconvert`, nbviewer
and a mailed `.ipynb` fall back to the page. If `anywidget` is not installed you
get the static layer alone and everything above still draws - except the one cell
that updates in place, which is what the widget is for.

> **This is the notebook you run.** It is committed with no outputs and should
> stay that way - each view's output is a whole page in an iframe, which adds a
> quarter of a megabyte per cell and renders as a blank on GitHub regardless.
> Clear outputs before committing.

```
pixi run notebook     # JupyterLab, on this file
pixi run marimo       # the same notebook, converted, in marimo
```

## This environment

> **If you are running this outside the repository's pixi environment:** gufe is
> not a dependency of the `alchemy-viz` wheel and is not installable from PyPI,
> where the only release is a 0.4 predating the 1.0 API. It comes from
> conda-forge:
>
> ```
> conda install -c conda-forge "gufe>=1.12"
> ```
>
> The fixture cells need none of that - a payload is a dict and drawing one is
> pure string assembly. Everything that takes a live object does.

In [ ]:
from alchemy_viz import __version__

try:
    import anywidget  # noqa: F401

    live = f"yes (anywidget {anywidget.__version__})"
except ImportError:
    live = "no - static pages only. `pip install alchemy-viz[notebook]`"

print(f"alchemy-viz {__version__}")
print(f"live widgets: {live}")
print()
for name, payload in payloads.items():
    print(f"  {name:24} {payload['type']}")

## openfe's classes are gufe's classes

OpenFE does not define its own classes for any of this. `openfe/__init__.py`
re-exports gufe's:

```python
from gufe import (
    AlchemicalNetwork, ChemicalSystem, Component, LigandAtomMapping,
    NonTransformation, ProteinComponent, ProteinMembraneComponent,
    SmallMoleculeComponent, SolvatedPDBComponent, SolventComponent, Transformation,
)
```

So `openfe.SmallMoleculeComponent` **is** `gufe.SmallMoleculeComponent` - the same
class object, not a subclass of it - and `alchemy_viz.payload_for` dispatches on
those gufe classes. That is the whole integration story: `view()` takes an openfe
object without alchemy-viz importing openfe or knowing that it exists. The cell
below is that claim, run.

In [ ]:
# openfe is not a dependency of this repository, so this cell reports rather than
# requires. The identity check is the claim above, run.
#
# gufe comes from conda-forge, never PyPI, so it can be absent even in a working
# alchemy-viz install. Going through require_gufe makes that case print the
# install instruction instead of a ModuleNotFoundError naming a package pip
# cannot get.
from alchemy_viz._gufe import require_gufe

gufe = require_gufe()

try:
    import openfe

    print(f"openfe {openfe.__version__}, gufe {gufe.__version__}")
    print()
    for name in (
        "SmallMoleculeComponent",
        "ProteinComponent",
        "SolventComponent",
        "ChemicalSystem",
        "LigandAtomMapping",
        "LigandNetwork",
        "Transformation",
        "AlchemicalNetwork",
    ):
        print(f"  openfe.{name:20} is gufe.{name:20} {getattr(openfe, name) is getattr(gufe, name)}")
except ImportError:
    print("openfe is not installed here, which is expected: this repository depends on")
    print("gufe alone. Install alchemy-viz into your openfe environment and the openfe")
    print("cells further up run as written.")
    print()
    print("    conda activate my-openfe-env")
    print("    pip install alchemy-viz[notebook]")

## What a view costs

Every displayed view sends the bundle to the browser: once in the page when
`static`, once in the widget's shell when live, and both when both. On
JupyterLab those messages share the kernel's iopub channel, which the server
rate-limits by default - so a notebook that creates many views in one burst can
have messages **dropped** rather than delivered slowly, and cells come up blank.
`static=False` and `live=False` are the two knobs.

In [ ]:
from alchemy_viz import bundle_source, shell_html

page = to_html(payloads["protein"])
shell = shell_html()

print(f"bundle                {len(bundle_source()):>10,} bytes")
print(f"payload (protein)     {len(json.dumps(payloads['protein'])):>10,} bytes")
print(f"static page           {len(page):>10,} bytes  <- per view, in the .ipynb")
print(f"widget shell          {len(shell):>10,} bytes  <- per view, over the comm")
print(f"both (the default)    {len(page) + len(shell):>10,} bytes")

## When it cannot draw

Four situations, four answers. Only the first is an exception, because only the
first is a mistake at the call site: `view()` was handed something that is not a
gufe object and not a dict, so there is nothing to serialize. The other three are
dicts, which Python passes through untouched - `to_html` does not validate - so
they reach the browser and fail there, and each gets a panel that says what is
wrong rather than leaving the cell blank.

**Two of these used to be one.** While some declared types had no view, a payload
could be perfectly well-formed and still undrawable, and the "no visualization
for X yet" panel was the case worth showing. Every type the schema declares now
has a view, so a *declared* type that will not draw is a malformed payload rather
than a missing view, and that panel is only reachable from outside the schema.

In [ ]:
# (a) something alchemy-viz has no view for at all: a mistake at the call site, so
#     it raises rather than drawing a panel about it
try:
    view(object())
except TypeError as e:
    print("TypeError:", e)

In [ ]:
# (b) a declared type whose body does not match the schema. This is what a
#     half-built payload looks like: the type is real and has a view, but
#     `gufe-key` and `smiles` are missing, so validation stops it before the
#     solvent view ever runs. The panel names the type and lists the fields.
view({"type": "SolventComponentViz", "name": "water"}, height="220px")

In [ ]:
# (c) a type nobody ever declared - a typo, or a payload from a build newer
#     than this one. A different panel: there is nothing to validate against,
#     so it says so and lists every type this build can draw.
view({"type": "NotAThing"}, height="220px")

In [ ]:
# (d) a dict that is not a payload at all. It never gets as far as a type, so
#     the panel cannot name one.
view({"name": "nameless"}, height="180px")

---

## Notes

**Why an iframe, and not the cell's own DOM.** Two properties of the bundle, not
caution: the `<gufe-*>` elements build light DOM, so a notebook's output-area CSS
would reach inside every view; and the engine loaders append a `<script>` to
`document.head` and read `window.$3Dmol` / `window.RDKit`, which in a notebook
page are the globals py3Dmol and nglview are already using, possibly at another
version. The iframe settles both for nothing.

**Engines come from a CDN.** RDKit and 3Dmol load on demand, from a view that
needs them - so a cell offline shows the page, the layout and the metadata, but
no depiction and no 3D viewer. `to_html(..., engines="bundled")` will close that
half once it exists.

**marimo.** `pixi run marimo` converts this file and opens it. The static layer
is plain HTML and needs nothing; the live layer goes through marimo's own
anywidget support.